In [1]:
!pip -q install gliner pandas numpy tqdm transformers accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.8/207.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 59.1 MB/s eta 0:00:00


In [2]:
import re
import gc
import json
import torch
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm.auto import tqdm
from google.colab import files
from gliner import GLiNER

pd.set_option("display.max_colwidth", 300)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)

OUTPUT_DIR = Path("/content/outputs_04_json_entidades_80_embargos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Carpeta de salida:", OUTPUT_DIR)

Dispositivo: cuda
Carpeta de salida: /content/outputs_04_json_entidades_80_embargos


In [3]:
uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("No se subió ningún archivo CSV.")

CSV_PATH = csv_files[0]
print("CSV cargado:", CSV_PATH)

Saving embargos_limpieza_avanzada_80_mejores.csv to embargos_limpieza_avanzada_80_mejores.csv
CSV cargado: embargos_limpieza_avanzada_80_mejores.csv


In [4]:
df = pd.read_csv(CSV_PATH)

print("Filas:", len(df))
print("Columnas:")
print(df.columns.tolist())

display(df.head(3))

Filas: 80
Columnas:
['id', 'nombre', 'clasificacion', 'texto_ocr', 'texto_limpio', 'texto_markdown', 'texto_base', 'texto_limpieza_avanzada', 'len_antes', 'len_despues', 'chars_eliminados', 'porcentaje_reduccion', 'palabras_antes', 'palabras_despues', 'porcentaje_reduccion_palabras', 'ocr_quality_score', 'ocr_estado', 'alerta_reduccion_alta', 'alerta_texto_muy_corto', 'alerta_posible_perdida_datos', 'len_chars', 'len_words', 'ratio_alpha', 'ratio_digits', 'ratio_rare_symbols', 'ratio_non_printable', 'line_breaks', 'one_letter_words', 'weird_alnum_tokens', 'repeated_char_tokens', 'near_empty_lines', 'symbol_only_lines', 'legal_terms_count', 'normal_word_ratio', 'important_data_count', 'alerta_desaparece_dni', 'alerta_desaparece_cuil_cuit', 'alerta_desaparece_cbu_cvu', 'alerta_desaparece_expediente', 'alerta_desaparece_monto', 'alerta_desaparece_fecha', 'alerta_desaparece_alias', 'alerta_desaparece_banco', 'alerta_desaparece_caratula', 'alerta_desaparece_juzgado_secretaria', 'antes_dni',

,id,nombre,clasificacion,texto_ocr,texto_limpio,texto_markdown,texto_base,texto_limpieza_avanzada,len_antes,len_despues,...,texto_normalizado_busqueda,largo_limpio,diferencia_largo,tiene_dni_regex,tiene_cuil_cuit_regex,tiene_cbu_cvu_regex,tiene_monto_regex,tiene_expediente_regex,tiene_palabras_clave_embargo,ranking_80_score
0,538118,Embargo - usuario,Embargo,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000000800 000000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr\n\n| Orgañismo: JUZGADO EN LO CIVIL Y COMERCIAL N0%16 - SA\n| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO\...,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000000800 000000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr\n\n| Orgañismo: JUZGADO EN LO CIVIL Y COMERCIAL N0%16 - SA\n| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO\...,# Documento legal\n\n## Metadatos\n\n* ID: 538118\n* Nombre: Embargo - usuario\n* Clasificación: Embargo\n\n## Texto limpio\n\n18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000000800 000000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203...,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000000800 000000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr\n\n| Orgañismo: JUZGADO EN LO CIVIL Y COMERCIAL N0%16 - SA\n| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO\...,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000800 000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr\n\n| Orgañismo: juzgado EN LO CIVIL Y COMERCIAL N0%16 - SA\n| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO\nNúmer...,6144,6124,...,18/2/46. 14:59 texto y datos de la notificacion - suprema corte de jus 5ncas/ y in | / / o 000000800 000000 vo) usuafio conectado: theiler pablo maximiliano - 203222152320 notr | organismo: juzgado en lo civil y comercial n0%16 - sa | chratula: o as sa c/ marquez na 02 ln e eu cutivo numero de c...,6144,3,True,True,True,True,True,True,150.8617
1,538084,Embargo - usuario,Embargo,">) 0\n\nPODER JUDICIAL DE LA NACION\nDIRECCIÓN GENERAL DE NOTIFICACIONES\n\nCARATULA DE NOTIFICACION\n\nOrigen: JUZGADO EN LO CIVIL Y COMERCIAL, N""11 - MAR DEL PL\nDestinatario: MERCADO PAGO ASSET MANAGEMENT S.A\nDirección: CASEROS AV. 3039 2\nCaracter: Común\nExpediente: MP-5712-2018\nCHIAMINO ...",">) 0\n\nPODER JUDICIAL DE LA NACION\nDIRECCIÓN GENERAL DE NOTIFICACIONES\n\nCARATULA DE NOTIFICACION\n\nOrigen: JUZGADO EN LO CIVIL Y COMERCIAL, N""11 - MAR DEL PL\nDestinatario: MERCADO PAGO ASSET MANAGEMENT S.A\nDirección: CASEROS AV. 3039 2\nCaracter: Común\nExpediente: MP-5712-2018\nCHIAMINO ...","# Documento legal\n\n## Metadatos\n\n* ID: 538084\n* Nombre: Embargo - usuario\n* Clasificación: Embargo\n\n## Texto limpio\n\n>) 0\n\nPODER JUDICIAL DE LA NACION\nDIRECCIÓN GENERAL DE NOTIFICACIONES\n\nCARATULA DE NOTIFICACION\n\nOrigen: JUZGADO EN LO CIVIL Y COMERCIAL, N""11 - MAR DEL PL\nDesti...",">) 0\n\nPODER JUDICIAL DE LA NACION\nDIRECCIÓN GENERAL DE NOTIFICACIONES\n\nCARATULA DE NOTIFICACION\n\nOrigen: JUZGADO EN LO CIVIL Y COMERCIAL, N""11 - MAR DEL PL\nDestinatario: MERCADO PAGO ASSET MANAGEMENT S.A\nDirección: CASEROS AV. 3039 2\nCaracter: Común\nExpediente: MP-5712-2018\nCHIAMINO ...",">) 0\n\nPODER JUDICIAL DE LA NACION\nDIRECCIÓN GENERAL DE NOTIFICACIONES\n\ncarátula DE NOTIFICACION\n\nOrigen: juzgado EN LO CIVIL Y COMERCIAL, N""11 - MAR DEL PL\nDestinatario: MERCADO PAGO ASSET MANAGEMENT S.A\nDirección: CASEROS AV. 3039 2\nCaracter: Común\nexpediente: MP-5712-2018\nCHIAMINO ...",5924,5876,...,">) 0 poder judicial de la nacion direccion general de notificaciones caratula de notificacion origen: juzgado en lo civil y comercial, n""11 - mar del pl destinatario: mercado pago asset management s.a direccion: c

In [5]:
def detectar_columna_texto(df):
    candidatos = [
        "texto_limpieza_avanzada",
        "texto_limpio_avanzado",
        "texto_limpio",
        "texto_base",
        "texto",
        "contenido",
    ]

    for col in candidatos:
        if col in df.columns:
            return col

    object_cols = df.select_dtypes(include=["object"]).columns.tolist()

    if not object_cols:
        raise ValueError("No se encontró columna de texto.")

    longitudes = {
        col: df[col].fillna("").astype(str).str.len().median()
        for col in object_cols
    }

    return max(longitudes, key=longitudes.get)


TEXT_COL = detectar_columna_texto(df)

if "id" in df.columns:
    ID_COL = "id"
else:
    ID_COL = "_row_id"
    df[ID_COL] = range(len(df))

NOMBRE_COL = "nombre" if "nombre" in df.columns else None
CLASIFICACION_COL = "clasificacion" if "clasificacion" in df.columns else None

df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)

print("Columna de texto:", TEXT_COL)
print("Columna ID:", ID_COL)
print("Columna nombre:", NOMBRE_COL)
print("Columna clasificación:", CLASIFICACION_COL)

df["chars"] = df[TEXT_COL].str.len()
df["palabras"] = df[TEXT_COL].str.split().str.len()

display(df[[ID_COL, TEXT_COL, "chars", "palabras"]].head())

Columna de texto: texto_limpieza_avanzada
Columna ID: id
Columna nombre: nombre
Columna clasificación: clasificacion


,id,texto_limpieza_avanzada,chars,palabras
0,538118,18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000800 000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr\n\n| Orgañismo: juzgado EN LO CIVIL Y COMERCIAL N0%16 - SA\n| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO\nNúmer...,6124,967
1,538084,">) 0\n\nPODER JUDICIAL DE LA NACION\nDIRECCIÓN GENERAL DE NOTIFICACIONES\n\ncarátula DE NOTIFICACION\n\nOrigen: juzgado EN LO CIVIL Y COMERCIAL, N""11 - MAR DEL PL\nDestinatario: MERCADO PAGO ASSET MANAGEMENT S.A\nDirección: CASEROS AV. 3039 2\nCaracter: Común\nexpediente: MP-5712-2018\nCHIAMINO ...",5876,951
2,495011,TEXTOJ[Y DATOS DE LA NOTIFICACIÓN\n\nUsuario co Hectado: DITIERI MARINA - 27289504937 QOnotificaciones.scba.govar |\n08) nismor] juzgado DE FAMILIA N* 3 - SAN MARTIN\ncarátula: RODINO MICAELA YANINA C/ VINCI BRIAN EMMANU OE ypeEntos\n| Número del[causa: 87462 ES libre\nTipo! de nolificación: OFI...,5866,923
3,534526,"23/2/26, 14:47 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUSTICIA - NOTIFICACIONES ELECTRÓNICAS\n\nscba.gov.ar\n\nPresentaciones y\nNi Notificaciones Electrónicas\nYT SUPREMA CORTE DE JUSTICIA\nPROVINCIA DE BUENOS AIRES\n\nIsuario Conectado:KAWATA KARINA ERICA - Acceso anterior: 17/12/...",5799,1062
4,542996,"(A O)\n""AA 0000 0\n\n00980397 7\n\nPODER JUDICIAL DE LA NACION\nDIRECCIÓN GENERAL DE NOTIFICACIONES\n\ncarátula DE NOTIFICACION\nOrigen: juzgado EN LO CIVIL Y COMERCIAL N*11 - MAR DBL PL\nDestinatario: MERCADO PAGO ASSET MANAGEMENT S.A\nDirección: CASEROS AV. 3039 *\nCaracter: Común\nexpediente:...",5746,922


In [6]:
MODEL_NAME = "urchade/gliner_multi_pii-v1"

print("Cargando modelo:", MODEL_NAME)

model = GLiNER.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)

tokenizer = model.data_processor.transformer_tokenizer

print("Modelo cargado:", MODEL_NAME)

Cargando modelo: urchade/gliner_multi_pii-v1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

Modelo cargado: urchade/gliner_multi_pii-v1


In [7]:
MAX_TOKENS_CHUNK = 300
OVERLAP_TOKENS = 50
PERSONA_THRESHOLD = 0.35

LABELS_PERSONA = ["persona"]

print("Max tokens chunk:", MAX_TOKENS_CHUNK)
print("Overlap tokens:", OVERLAP_TOKENS)
print("Threshold persona:", PERSONA_THRESHOLD)

Max tokens chunk: 300
Overlap tokens: 50
Threshold persona: 0.35


In [8]:
def contar_tokens(texto, tokenizer):
    if not isinstance(texto, str):
        texto = str(texto)

    try:
        return len(tokenizer.encode(texto, add_special_tokens=False))
    except Exception:
        return np.nan


def crear_chunks_por_tokens(texto, tokenizer, max_tokens=300, overlap_tokens=50):
    if not isinstance(texto, str):
        texto = str(texto)

    texto = texto.strip()

    if not texto:
        return []

    if overlap_tokens >= max_tokens:
        raise ValueError("overlap_tokens debe ser menor que max_tokens.")

    token_ids = tokenizer.encode(texto, add_special_tokens=False)

    chunks = []
    start = 0
    chunk_id = 0

    while start < len(token_ids):
        end = min(start + max_tokens, len(token_ids))
        token_slice = token_ids[start:end]

        chunk_text = tokenizer.decode(token_slice, skip_special_tokens=True)

        chunks.append({
            "chunk_id": chunk_id,
            "token_start": start,
            "token_end": end,
            "n_tokens_chunk": len(token_slice),
            "texto_chunk": chunk_text,
        })

        if end >= len(token_ids):
            break

        start = end - overlap_tokens
        chunk_id += 1

    return chunks


def limpiar_nombre_persona(nombre):
    if not isinstance(nombre, str):
        return ""

    nombre = nombre.strip()

    # Quitar saltos raros
    nombre = re.sub(r"\s+", " ", nombre)

    # Quitar cargos o prefijos pegados al inicio
    nombre = re.sub(
        r"^(Dr\.?|Dra\.?|Juez\.?|Jueza\.?|Secretario\.?|Secretaria\.?|Fdo\.?)\s+",
        "",
        nombre,
        flags=re.IGNORECASE
    )

    # Quitar cargos pegados al final
    nombre = re.sub(
        r"\s+(JUEZ|JUEZA|SECRETARIO|SECRETARIA|LETRADO|LETRADA)$",
        "",
        nombre,
        flags=re.IGNORECASE
    )

    nombre = nombre.strip(" .,:;-_")

    return nombre


def persona_valida(nombre):
    if not isinstance(nombre, str):
        return False

    nombre = limpiar_nombre_persona(nombre)

    if len(nombre) < 5:
        return False

    # Evitar cosas que claramente no son personas
    patrones_invalidos = [
        r"\bDNI\b",
        r"\bCUIL\b",
        r"\bCUIT\b",
        r"\bCBU\b",
        r"\bCVU\b",
        r"http",
        r"www",
        r"notificacion",
        r"notificaciones",
        r"scba",
        r"mercado\s*pago",
        r"banco",
        r"juzgado",
        r"tribunal",
    ]

    for patron in patrones_invalidos:
        if re.search(patron, nombre, flags=re.IGNORECASE):
            return False

    # Debe tener letras
    if not re.search(r"[A-Za-zÁÉÍÓÚáéíóúÑñ]", nombre):
        return False

    # Evitar textos con demasiados números
    digitos = len(re.findall(r"\d", nombre))
    if digitos > 0:
        return False

    return True


def extraer_personas_gliner(texto, doc_id):
    entidades = []

    chunks = crear_chunks_por_tokens(
        texto,
        tokenizer,
        max_tokens=MAX_TOKENS_CHUNK,
        overlap_tokens=OVERLAP_TOKENS
    )

    for chunk in chunks:
        chunk_text = chunk["texto_chunk"]

        try:
            predicciones = model.predict_entities(
                chunk_text,
                LABELS_PERSONA,
                threshold=PERSONA_THRESHOLD
            )
        except Exception as e:
            print(f"Error en doc {doc_id}, chunk {chunk['chunk_id']}: {e}")
            continue

        for ent in predicciones:
            texto_original = ent.get("text", "").strip()
            texto_limpio = limpiar_nombre_persona(texto_original)

            if not persona_valida(texto_limpio):
                continue

            entidades.append({
                "id": doc_id,
                "texto": texto_limpio,
                "texto_original": texto_original,
                "etiqueta": "persona",
                "metodo": "gliner",
                "modelo": MODEL_NAME,
                "score": ent.get("score", None),
                "chunk_id": chunk["chunk_id"],
                "start": ent.get("start", None),
                "end": ent.get("end", None),
            })

    return entidades

In [9]:
REGEX_ENTIDADES = {
    "DNI": re.compile(
        r"\b(?:DNI|D\.N\.I\.|DOCUMENTO|DOC\.?)\s*(?:N[°º\*ro\.]*\s*)?[:\-]?\s*(\d{1,2}[\.\s]?\d{3}[\.\s]?\d{3}|\d{7,8})\b",
        flags=re.IGNORECASE
    ),

    "CUIL": re.compile(
        r"\b(?:CUIL|C\.U\.I\.L\.)\s*[:\-]?\s*((?:20|23|24|27)[\-\s]?\d{8}[\-\s]?\d)\b",
        flags=re.IGNORECASE
    ),

    "CUIT": re.compile(
        r"\b(?:CUIT|C\.U\.I\.T\.)\s*[:\-]?\s*((?:30|33|34)[\-\s]?\d{8}[\-\s]?\d)\b",
        flags=re.IGNORECASE
    ),

    "CBU": re.compile(
        r"\bCBU\s*(?:N[°º\*ro\.]*\s*)?[:\-]?\s*((?:\d[\s\-]?){22})\b",
        flags=re.IGNORECASE
    ),

    "CVU": re.compile(
        r"\bCVU\s*(?:N[°º\*ro\.]*\s*)?[:\-]?\s*((?:\d[\s\-]?){22})\b",
        flags=re.IGNORECASE
    ),
}


def normalizar_numero(valor):
    if not isinstance(valor, str):
        return ""

    return re.sub(r"\D", "", valor)


def reconstruir_texto_entidad(etiqueta, valor_original):
    valor_original = re.sub(r"\s+", " ", str(valor_original)).strip()

    return f"{etiqueta} {valor_original}"


def extraer_entidades_regex(texto, doc_id):
    entidades = []

    if not isinstance(texto, str):
        texto = str(texto)

    for etiqueta, patron in REGEX_ENTIDADES.items():
        for match in patron.finditer(texto):
            valor_original = match.group(1).strip()
            valor_normalizado = normalizar_numero(valor_original)

            # Validaciones por tipo
            if etiqueta == "DNI" and not re.fullmatch(r"\d{7,8}", valor_normalizado):
                continue

            if etiqueta in ["CUIL", "CUIT"] and not re.fullmatch(r"\d{11}", valor_normalizado):
                continue

            if etiqueta in ["CBU", "CVU"] and not re.fullmatch(r"\d{22}", valor_normalizado):
                continue

            entidades.append({
                "id": doc_id,
                "texto": reconstruir_texto_entidad(etiqueta, valor_original),
                "valor_normalizado": valor_normalizado,
                "etiqueta": etiqueta,
                "metodo": "regex",
                "modelo": None,
                "score": None,
                "chunk_id": None,
                "start": match.start(),
                "end": match.end(),
            })

    return entidades

In [10]:
idx_prueba = 0

row = df.iloc[idx_prueba]
doc_id = row[ID_COL]
texto = str(row[TEXT_COL])

print("ID:", doc_id)
print("Caracteres:", len(texto))
print("Tokens:", contar_tokens(texto, tokenizer))
print(texto[:1000])

ID: 538118
Caracteres: 6124
Tokens: 1918
18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y

in | / /
o 000800 000 vo)

Usuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr

| Orgañismo: juzgado EN LO CIVIL Y COMERCIAL N0%16 - SA
| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO
Número de causa: s/prrse-2093 o
Tipo de notificación: Jeeps ELECTRONICO 4 paar
Destinatarios: A Y
Fécha notificación: 20/2/2026 " O.
Alta a disponibilidad 19/2/2026 10:44:34
| Fifmal digital: Firma valida
Firmablo y Notificado por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026
10:44.33
Fifmabo por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026

2]¡BRANDARIZ Luciana Rocio. --- Certificado Correcto. Fecha de Firma:
2:20p6 32:44.42 THEILER Pablo Maximiliano. --- Certificado Correcto. Fecha de
Firma: 14/02/2026 14:30:16

OFICIO LEY 22.172

RECIBIDO

MERCADOLIBRE SRL Domicilio: AV. CASEROS N* 3039, PISO N* 

In [11]:
todas_entidades = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    doc_id = row[ID_COL]
    texto = str(row[TEXT_COL])

    entidades_persona = extraer_personas_gliner(texto, doc_id)
    entidades_regex = extraer_entidades_regex(texto, doc_id)

    todas_entidades.extend(entidades_persona)
    todas_entidades.extend(entidades_regex)

df_entidades = pd.DataFrame(todas_entidades)

print("Total entidades brutas:", len(df_entidades))

display(df_entidades.head(20))

  0%|          | 0/80 [00:00<?, ?it/s]

Total entidades brutas: 1821


,id,texto,texto_original,etiqueta,metodo,modelo,score,chunk_id,start,end,valor_normalizado
0,538118,THEILER Pablo Maximiliano,THEILER Pablo Maximiliano,persona,gliner,urchade/gliner_multi_pii-v1,0.996411,0.0,125,150,NaN
1,538118,PETRONE Maria Teresa,PETRONE Maria Teresa,persona,gliner,urchade/gliner_multi_pii-v1,0.996245,0.0,511,531,NaN
2,538118,PETRONE Maria Teresa,PETRONE Maria Teresa,persona,gliner,urchade/gliner_multi_pii-v1,0.997993,0.0,613,633,NaN
3,538118,BRANDARIZ Luciana Rocio,BRANDARIZ Luciana Rocio,persona,gliner,urchade/gliner_multi_pii-v1,0.868619,0.0,696,719,NaN
4,538118,THEILER Pablo Maximiliano,THEILER Pablo Maximiliano,persona,gliner,urchade/gliner_multi_pii-v1,0.997906,0.0,779,804,NaN
5,538118,THEILER Pablo Maximiliano,THEILER Pablo Maximiliano,persona,gliner,urchade/gliner_multi_pii-v1,0.810957,1.0,32,57,NaN
6,538118,NAHUEL OSCAR,NAHUEL OSCAR,persona,gliner,urchade/gliner_multi_pii-v1,0.406604,1.0,398,410,NaN
7,538118,María Taresa Petrone,Dra. María Taresa Petrone,persona,gliner,urchade/gliner_multi_pii-v1,0.948614,1.0,562,587,NaN
8,538118,Juan Andrés Gasparini,Dr. Juan Andrés Gasparini,persona,gliner,urchade/gliner_multi_pii-v1,0.887384,1.0,618,643,NaN
9,538118,NAHUEL OSCAR MARQUEZ,NAHUEL OSCAR MARQUEZ,persona,gliner,urchade/gliner_multi_pii-v1,0.998214,2.0,185,205,NaN


In [12]:
def clave_dedupe(row):
    texto = str(row["texto"]).lower().strip()
    texto = re.sub(r"\s+", " ", texto)
    return f"{row['id']}|{row['etiqueta']}|{texto}"


if len(df_entidades) > 0:
    df_entidades["clave_dedupe"] = df_entidades.apply(clave_dedupe, axis=1)

    antes = len(df_entidades)

    df_entidades = (
        df_entidades
        .sort_values(["id", "etiqueta", "score"], ascending=[True, True, False], na_position="last")
        .drop_duplicates("clave_dedupe", keep="first")
        .drop(columns=["clave_dedupe"])
        .reset_index(drop=True)
    )

    despues = len(df_entidades)

    print("Entidades antes de deduplicar:", antes)
    print("Entidades después de deduplicar:", despues)

display(df_entidades.head(30))

Entidades antes de deduplicar: 1821
Entidades después de deduplicar: 1377


,id,texto,texto_original,etiqueta,metodo,modelo,score,chunk_id,start,end,valor_normalizado
0,494151,CUIT 30-70308853-4,NaN,CUIT,regex,None,NaN,NaN,2881,2899,30703088534
1,494151,DNI 17.196.196,NaN,DNI,regex,None,NaN,NaN,1549,1563,17196196
2,494151,JOSÉ LUIS VALENTINI,JOSÉ LUIS VALENTINI,persona,gliner,urchade/gliner_multi_pii-v1,0.999420,2.0,46,65,NaN
3,494151,Leticia Mabel Frappa,Dra. Leticia Mabel Frappa,persona,gliner,urchade/gliner_multi_pii-v1,0.976097,8.0,324,349,NaN
4,494151,LUÍS VALENTIN,LUÍS VALENTIN,persona,gliner,urchade/gliner_multi_pii-v1,0.933755,4.0,254,267,NaN
5,494151,FRAPPA LETICIA MABEL,FRAPPA LETICIA MABEL,persona,gliner,urchade/gliner_multi_pii-v1,0.926813,0.0,137,157,NaN
6,494151,"MARANO, Agustin","Dr. MARANO, Agustin",persona,gliner,urchade/gliner_multi_pii-v1,0.893285,3.0,328,347,NaN
7,494151,BALDE,BALDE,persona,gliner,urchade/gliner_multi_pii-v1,0.829039,11.0,427,432,NaN
8,494151,MORETTO GALASTRO Mariel Tamara,MORETTO GALASTRO Mariel Tamara,persona,gliner,urchade/gliner_multi_pii-v1,0.738200,0.0,518,548,NaN
9,494151,ERAPPA Leticia Mabel,ERAPPA Leticia Mabel,persona,gliner,urchade/gliner_multi_pii-v1,0.706310,0.0,622,642,NaN


In [13]:
print("Total entidades:", len(df_entidades))

resumen_etiquetas = (
    df_entidades
    .groupby("etiqueta")
    .size()
    .reset_index(name="cantidad")
    .sort_values("cantidad", ascending=False)
)

display(resumen_etiquetas)

resumen_docs = (
    df_entidades
    .groupby("id")
    .agg(
        total_entidades=("texto", "count"),
        etiquetas_encontradas=("etiqueta", lambda x: ", ".join(sorted(set(x))))
    )
    .reset_index()
    .sort_values("total_entidades", ascending=False)
)

display(resumen_docs.head(20))

Total entidades: 1377


,etiqueta,cantidad
5,persona,1128
4,DNI,129
0,CBU,69
2,CUIT,36
1,CUIL,12
3,CVU,3


,id,total_entidades,etiquetas_encontradas
14,507932,51,"CBU, DNI, persona"
13,507931,43,"CBU, DNI, persona"
30,525055,40,"CBU, DNI, persona"
34,529847,29,"CBU, CUIL, DNI, persona"
78,547373,28,"CUIT, DNI, persona"
20,516903,25,"CBU, CUIT, DNI, persona"
9,501211,25,persona
55,536742,25,"CBU, CUIT, DNI, persona"
74,546334,24,"CBU, persona"
43,536705,23,"CBU, DNI, persona"


In [14]:
{
  "id": "...",
  "clasificacion": "...",
  "nombre": "...",
  "texto_limpio": "...",
  "entidades": [
    {
      "texto": "...",
      "etiqueta": "..."
    }
  ],
  "total_entidades": 0,
  "etiquetas_encontradas": []
}

{'id': '...',
 'clasificacion': '...',
 'nombre': '...',
 'texto_limpio': '...',
 'entidades': [{'texto': '...', 'etiqueta': '...'}],
 'total_entidades': 0,
 'etiquetas_encontradas': []}

In [15]:
documentos_json = []

for _, row in df.iterrows():
    doc_id = row[ID_COL]
    texto_limpio = str(row[TEXT_COL])

    clasificacion = str(row[CLASIFICACION_COL]) if CLASIFICACION_COL else ""
    nombre = str(row[NOMBRE_COL]) if NOMBRE_COL else ""

    entidades_doc = df_entidades[df_entidades["id"] == doc_id].copy()

    entidades_lista = []

    for _, ent in entidades_doc.iterrows():
        entidades_lista.append({
            "texto": str(ent["texto"]),
            "etiqueta": str(ent["etiqueta"]),
        })

    etiquetas_encontradas = sorted(list(set([e["etiqueta"] for e in entidades_lista])))

    documento = {
        "id": str(doc_id),
        "clasificacion": clasificacion,
        "nombre": nombre,
        "texto_limpio": texto_limpio,
        "entidades": entidades_lista,
        "total_entidades": len(entidades_lista),
        "etiquetas_encontradas": etiquetas_encontradas,
    }

    documentos_json.append(documento)

print("Documentos JSON:", len(documentos_json))
print("Primer documento:")
print(json.dumps(documentos_json[0], ensure_ascii=False, indent=2)[:3000])

Documentos JSON: 80
Primer documento:
{
  "id": "538118",
  "clasificacion": "Embargo",
  "nombre": "Embargo - usuario",
  "texto_limpio": "18/2/46. 14:59 TEXTO Y DATOS DE LA NOTIFICACIÓN - SUPREMA CORTE DE JUS 5ncas/ y\n\nin | / /\no 000800 000 vo)\n\nUsuafio conectado: THEILER Pablo Maximiliano - 203222152320 notr\n\n| Orgañismo: juzgado EN LO CIVIL Y COMERCIAL N0%16 - SA\n| Chrátula: o AS SA C/ MARQUEZ NA 02 Ln e eu CUTIVO\nNúmero de causa: s/prrse-2093 o\nTipo de notificación: Jeeps ELECTRONICO 4 paar\nDestinatarios: A Y\nFécha notificación: 20/2/2026 \" O.\nAlta a disponibilidad 19/2/2026 10:44:34\n| Fifmal digital: Firma valida\nFirmablo y Notificado por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026\n10:44.33\nFifmabo por: PETRONE Maria Teresa. JUEZ --- Certificado Correcto. Fecha de Firma: 19/02/2026\n\n2]¡BRANDARIZ Luciana Rocio. --- Certificado Correcto. Fecha de Firma:\n2:20p6 32:44.42 THEILER Pablo Maximiliano. --- Certificado Correcto. Fec

In [16]:
JSON_PATH = OUTPUT_DIR / "embargos_80_entidades_persona_regex.json"
JSONL_PATH = OUTPUT_DIR / "embargos_80_entidades_persona_regex.jsonl"
CSV_ENTIDADES_PATH = OUTPUT_DIR / "entidades_embargos_80_persona_regex.csv"
CSV_RESUMEN_PATH = OUTPUT_DIR / "resumen_embargos_80_persona_regex.csv"

# JSON normal
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(documentos_json, f, ensure_ascii=False, indent=2)

# JSONL
with open(JSONL_PATH, "w", encoding="utf-8") as f:
    for doc in documentos_json:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

# CSV detallado
df_entidades.to_csv(CSV_ENTIDADES_PATH, index=False, encoding="utf-8-sig")

# CSV resumen por documento
df_resumen_json = pd.DataFrame([
    {
        "id": doc["id"],
        "clasificacion": doc["clasificacion"],
        "nombre": doc["nombre"],
        "total_entidades": doc["total_entidades"],
        "etiquetas_encontradas": ", ".join(doc["etiquetas_encontradas"]),
        "chars_texto": len(doc["texto_limpio"]),
    }
    for doc in documentos_json
])

df_resumen_json.to_csv(CSV_RESUMEN_PATH, index=False, encoding="utf-8-sig")

print("Archivos guardados:")
print(JSON_PATH)
print(JSONL_PATH)
print(CSV_ENTIDADES_PATH)
print(CSV_RESUMEN_PATH)

Archivos guardados:
/content/outputs_04_json_entidades_80_embargos/embargos_80_entidades_persona_regex.json
/content/outputs_04_json_entidades_80_embargos/embargos_80_entidades_persona_regex.jsonl
/content/outputs_04_json_entidades_80_embargos/entidades_embargos_80_persona_regex.csv
/content/outputs_04_json_entidades_80_embargos/resumen_embargos_80_persona_regex.csv


In [17]:
print("Cantidad de documentos:", len(documentos_json))
print("Cantidad de entidades:", len(df_entidades))

print("\nEntidades por etiqueta:")
display(resumen_etiquetas)

print("\nResumen por documento:")
display(df_resumen_json.head(20))

docs_sin_entidades = df_resumen_json[df_resumen_json["total_entidades"] == 0]
print("\nDocumentos sin entidades:", len(docs_sin_entidades))

if len(docs_sin_entidades) > 0:
    display(docs_sin_entidades)

Cantidad de documentos: 80
Cantidad de entidades: 1377

Entidades por etiqueta:


,etiqueta,cantidad
5,persona,1128
4,DNI,129
0,CBU,69
2,CUIT,36
1,CUIL,12
3,CVU,3



Resumen por documento:


,id,clasificacion,nombre,total_entidades,etiquetas_encontradas,chars_texto
0,538118,Embargo,Embargo - usuario,17,"CBU, CUIT, DNI, persona",6124
1,538084,Embargo,Embargo - usuario,15,"CBU, CUIL, DNI, persona",5876
2,495011,Embargo,Embargo - usuario,21,"CBU, CUIL, DNI, persona",5866
3,534526,Embargo,Embargo - Empleado,13,"CBU, DNI, persona",5799
4,542996,Embargo,Embargo - usuario,12,"CBU, CUIL, DNI, persona",5746
5,538081,Embargo,Embargo - usuario,14,"CBU, CUIL, DNI, persona",5697
6,524057,Embargo,Embargo - usuario,13,"CBU, DNI, persona",5688
7,536725,Embargo,Embargo - usuario,16,"CBU, CUIT, DNI, persona",5682
8,536741,Embargo,Embargo - usuario,22,"CBU, CUIT, DNI, persona",5445
9,524053,Embargo,Embargo - usuario,14,"CBU, CUIT, DNI, persona",5421



Documentos sin entidades: 0


In [18]:
files.download(str(JSON_PATH))
files.download(str(CSV_ENTIDADES_PATH))
files.download(str(CSV_RESUMEN_PATH))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusión preliminar

Se procesaron 80 embargos seleccionados desde `embargos_limpieza_avanzada_80_mejores.csv`.

La entidad `persona` fue extraída utilizando el modelo `urchade/gliner_multi_pii-v1`, que en pruebas anteriores resultó más conveniente que `gliner-community/gliner_medium-v2.5` por generar menos ruido y detectar entidades personales de forma más precisa.

Para evitar truncamiento en documentos largos, se utilizó una estrategia de chunking con fragmentos de 300 tokens y overlap de 50 tokens.

Las entidades `DNI`, `CUIL`, `CUIT`, `CBU` y `CVU` fueron extraídas mediante expresiones regulares aplicadas directamente sobre el texto limpio de cada embargo. Esta estrategia permite capturar datos estructurados de manera más controlada que un modelo de lenguaje.

Finalmente, se generaron tres salidas principales:
- JSON con la estructura requerida para documentos y entidades.
- CSV detallado con una fila por entidad.
- CSV resumen con cantidad de entidades por documento.

Estos archivos constituyen una base preliminar de entidades candidatas. Para considerarlas etiquetas verdaderas, se recomienda realizar una revisión manual supervisada.